# Gut Bacteria Project: Prediction & Attention Analysis

This Jupyter Notebook demonstrates how to run the pipeline to predict drug-bacteria interactions and visualize the attention weights of the Multi-Modal Model.

## 1. Check Input Files
Ensure `sample1.txt` is present and the corresponding `.fasta` file is inside the `input/` directory.

In [1]:
import os

input_txt = 'sample1.txt'
print("--- Contents of sample1.txt ---")
with open(input_txt, 'r') as f:
    print(f.read())

--- Contents of sample1.txt ---
drug: Diltiazem
drug smiles: COc1ccc(C2Sc3ccccc3N(CC[NH+](C)C)C(=O)C2OC(C)=O)cc1
bacterial strain name: Bacteroides thetaiotaomicron VPI-5482

target protein: WP_008760980.1


## 2. Run the Prediction Pipeline
We will run `prediction.py` with `sample1.txt`. `sample1.txt` contains the line `target protein: WP_008760980.1`, so this protein is used as a user-defined protein: it is added to the model input, and its EC number and atom-attention map are reported in addition to the EC enrichment analysis. (The same can be done with the `--target_protein` / `--include_proteins` options.)

*(Note: The first time you run this, KofamScan and ESM-C downloading may take a few minutes. Subsequent runs will use cached results.)*

In [ ]:
!python prediction.py --input_txt sample1.txt --cpu_cores 12

## 3. View Attention-Based Interpretation
Once the prediction finishes successfully, the results for each drug-bacteria pair are saved in `attention analysis/<drug>_<bacterial strain>/`:
- `EC_candidates_top10pct.csv`: representative EC candidates (FE > 1 and P <= 0.05 in the top 10% proteins, one-sided Fisher's exact test) with their protein IDs and descriptions
- `EC_candidate_proteins_top10pct.csv`: one row per (EC class, protein) with the paths of the atom-attention maps
- `EC_atom_maps/EC_<EC>/`: EC-averaged (`*_mean.svg`) and protein-level atom-attention maps
- `user_defined_proteins.csv` / `atom_attention_user_*.svg`: EC number and atom-attention map of each user-defined protein

In [ ]:
import pandas as pd
from IPython.display import Image, display, SVG
import glob, os

for pair_dir in sorted(glob.glob('attention analysis/*/')):
    print(f"===== {pair_dir} =====")
    for csv_name in ['EC_candidates_top10pct.csv', 'user_defined_proteins.csv']:
        csv_path = os.path.join(pair_dir, csv_name)
        if os.path.exists(csv_path):
            print(csv_name)
            display(pd.read_csv(csv_path))
    # EC-averaged maps (one per representative EC) and user-defined protein maps;
    # protein-level maps of each EC class are in EC_atom_maps/EC_<EC>/
    images = sorted(glob.glob(os.path.join(pair_dir, 'EC_atom_maps', '*', '*_mean.svg'))) \
           + sorted(glob.glob(os.path.join(pair_dir, 'atom_attention_user_*.svg'))) \
           + sorted(glob.glob(os.path.join(pair_dir, '*.png')))
    for img in images:
        print(f"Displaying: {img}")
        if img.lower().endswith('.svg'):
            display(SVG(filename=img))
        else:
            display(Image(filename=img))